In [4]:
!pip install z3-solver==4.12.2.0

In [5]:
# ============================================================
# FILTER EXPERIMENT 03 — CLEAN STANDALONE
# Negative gradient on rejected claims vs hard filter
# vs contaminated
# MIT-compatible. Zero proprietary references.
# ============================================================

import os
import json
import random
import hashlib
from enum import Enum

import numpy as np
import torch
import torch.nn.functional as F
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# CONFIGURATION
# ============================================================

EXPERIMENT = "Filter Experiment 03"

SEEDS = [11, 22, 33]
EPOCHS = 4
UPDATES_PER_EPOCH = 84

LR = 5e-5
MAX_LENGTH = 64
GRAD_CLIP = 1.0
BETA_EPISTEMIC = 1.0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = "/kaggle/working/filter_exp03"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using device:", DEVICE)


# ============================================================
# EXPECTED MODEL FILE HASHES
# ============================================================

EXPECTED_HASHES = {
    "model.safetensors": (
        "c7d00560d8910fbed77ffad4065dee5011c41ba401b1064e749c498ba9e20373"
    ),
    "config.json": (
        "337e7106d8f04da30d6ef1617faa51369cddaa34a855bf76844c9798b46b26e8"
    ),
    "vocab.json": (
        "f6bd25a65e4e63ca31360e9fb11c7e4f9a391a78385d640acd814092dd6eee4f"
    ),
    "merges.txt": (
        "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    ),
}


# ============================================================
# DATA
# ============================================================

GROUND_TRUTH = {
    ("entity_a", "color"): "blue",
    ("entity_b", "color"): "green",
    ("entity_c", "color"): "yellow",
    ("entity_d", "color"): "purple",
    ("entity_e", "shape"): "circle",
    ("entity_f", "shape"): "square",
    ("entity_g", "shape"): "triangle",
    ("entity_h", "shape"): "hexagon",
    ("entity_i", "code"): "17",
    ("entity_j", "code"): "29",
    ("entity_k", "code"): "41",
    ("entity_l", "code"): "53",
}

SYNTHETIC_DATA = [
    {
        "subject": "entity_c",
        "predicate": "color",
        "value": "yellow",
    },
    {
        "subject": "entity_d",
        "predicate": "color",
        "value": "purple",
    },
    {
        "subject": "entity_g",
        "predicate": "shape",
        "value": "triangle",
    },
    {
        "subject": "entity_h",
        "predicate": "shape",
        "value": "hexagon",
    },
    {
        "subject": "entity_k",
        "predicate": "code",
        "value": "41",
    },
    {
        "subject": "entity_l",
        "predicate": "code",
        "value": "53",
    },
    {
        "subject": "entity_a",
        "predicate": "color",
        "value": "red",
    },
    {
        "subject": "entity_b",
        "predicate": "color",
        "value": "orange",
    },
    {
        "subject": "entity_e",
        "predicate": "shape",
        "value": "triangle",
    },
    {
        "subject": "entity_f",
        "predicate": "shape",
        "value": "circle",
    },
    {
        "subject": "entity_i",
        "predicate": "code",
        "value": "99",
    },
    {
        "subject": "entity_j",
        "predicate": "code",
        "value": "88",
    },
    {
        "subject": "entity_x",
        "predicate": "color",
        "value": "silver",
    },
    {
        "subject": "entity_y",
        "predicate": "shape",
        "value": "star",
    },
]

CONTAMINATION_TEST = [
    ("entity_a", "color", "blue", "red"),
    ("entity_b", "color", "green", "orange"),
    ("entity_e", "shape", "circle", "triangle"),
    ("entity_f", "shape", "square", "circle"),
    ("entity_i", "code", "17", "99"),
    ("entity_j", "code", "29", "88"),
]


# ============================================================
# FILTER TYPES
# ============================================================

class Claim(BaseModel):
    subject: str
    predicate: str
    value: str


class Verdict(str, Enum):
    VERIFIED = "VERIFIED"
    CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"
    INVALID = "INVALID"


class FilterGate:
    def __init__(self, ground_truth):
        self.gt = {
            (
                subject.strip().lower(),
                predicate.strip().lower(),
            ): str(value).strip().lower()
            for (subject, predicate), value in ground_truth.items()
        }

    def audit(self, raw):
        try:
            claim = Claim(**raw)
        except Exception:
            return Verdict.INVALID

        key = (
            claim.subject.strip().lower(),
            claim.predicate.strip().lower(),
        )

        value = claim.value.strip().lower()

        if key not in self.gt:
            return Verdict.UNKNOWN

        if value == self.gt[key]:
            return Verdict.VERIFIED

        return Verdict.CONTRADICTED


# ============================================================
# UTILITIES
# ============================================================

def fact_text(record):
    return (
        f"FACT: {record['subject']} "
        f"{record['predicate']} = {record['value']}"
    )


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def cycle_to_length(items, length, seed):
    if not items:
        raise ValueError(
            "Cannot create a training stream from an empty list."
        )

    output = []

    while len(output) < length:
        output.extend(items)

    output = output[:length]
    random.Random(seed).shuffle(output)

    return output


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def continuation_score(model, prefix, continuation):
    model.eval()

    prefix_ids = tokenizer(
        prefix,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)

    full_ids = tokenizer(
        prefix + continuation,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)

    logits = model(input_ids=full_ids).logits[:, :-1, :]
    labels = full_ids[:, 1:]

    log_probs = F.log_softmax(logits, dim=-1)

    selected = torch.gather(
        log_probs,
        dim=-1,
        index=labels.unsqueeze(-1),
    ).squeeze(-1)

    continuation_start = max(prefix_ids.shape[1] - 1, 0)
    continuation_log_probs = selected[:, continuation_start:]

    if continuation_log_probs.numel() == 0:
        raise RuntimeError(
            "No continuation tokens were available for evaluation."
        )

    return float(
        continuation_log_probs.mean().item()
    )


def evaluate(model, label):
    rows = []

    for subject, predicate, truth, false_value in CONTAMINATION_TEST:
        prefix = f"FACT: {subject} {predicate} ="

        truth_score = continuation_score(
            model,
            prefix,
            " " + truth,
        )

        false_score = continuation_score(
            model,
            prefix,
            " " + false_value,
        )

        truth_margin = float(truth_score - false_score)

        rows.append({
            "branch": label,
            "subject": subject,
            "truth_margin": truth_margin,
            "prefers_truth": bool(truth_margin > 0),
        })

    return rows


# ============================================================
# TRAINING LOSSES
# ============================================================

def causal_ce(logits, input_ids):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
    )


def sequence_mean_logprob(logits, input_ids):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    log_probs = F.log_softmax(
        shift_logits,
        dim=-1,
    )

    selected = torch.gather(
        log_probs,
        dim=2,
        index=shift_labels.unsqueeze(-1),
    ).squeeze(-1)

    return selected.mean()


# ============================================================
# TRAINING
# ============================================================

def train_branch(branch, stream, seed):
    set_seed(seed)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
    ).to(DEVICE)

    # No se necesita KV cache durante entrenamiento.
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
    )

    for epoch in range(EPOCHS):
        epoch_losses = []

        for record in stream:
            encoded = tokenizer(
                record["text"],
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(**encoded).logits

            if branch in {"CONTAMINATED", "HARD_FILTER"}:
                loss = causal_ce(
                    logits,
                    encoded["input_ids"],
                )

            elif branch == "EPISTEMIC":
                if record["verdict"] == Verdict.VERIFIED.value:
                    loss = causal_ce(
                        logits,
                        encoded["input_ids"],
                    )
                else:
                    # Gradient descent minimizes the log-probability
                    # of rejected or unknown claims.
                    loss = (
                        BETA_EPISTEMIC
                        * sequence_mean_logprob(
                            logits,
                            encoded["input_ids"],
                        )
                    )

            else:
                raise ValueError(
                    f"Unknown training branch: {branch}"
                )

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"{branch}: non-finite loss at "
                    f"epoch {epoch + 1}: {loss.detach().item()}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            epoch_losses.append(
                float(loss.detach().cpu().item())
            )

        mean_epoch_loss = float(np.mean(epoch_losses))

        print(
            f"Seed {seed} | "
            f"{branch} | "
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Loss {mean_epoch_loss:.6f}"
        )

    results = evaluate(model, branch)

    # El optimizador mantiene referencias a los parámetros.
    del optimizer
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# ============================================================
# LOCATE MODEL AND TOKENIZER
# ============================================================

model_candidates = []
tokenizer_candidates = []

for root, _, files in os.walk("/kaggle/input"):
    file_set = set(files)

    if {
        "config.json",
        "model.safetensors",
    }.issubset(file_set):
        model_candidates.append(root)

    if {
        "vocab.json",
        "merges.txt",
    }.issubset(file_set):
        tokenizer_candidates.append(root)

if not model_candidates:
    raise FileNotFoundError(
        "No model directory containing config.json and "
        "model.safetensors was found under /kaggle/input."
    )

if not tokenizer_candidates:
    raise FileNotFoundError(
        "No tokenizer directory containing vocab.json and "
        "merges.txt was found under /kaggle/input."
    )

MODEL_PATH = model_candidates[0]
TOKENIZER_PATH = tokenizer_candidates[0]

print("Model path:", MODEL_PATH)
print("Tokenizer path:", TOKENIZER_PATH)


# ============================================================
# VERIFY FILE HASHES
# ============================================================

for filename, expected_hash in EXPECTED_HASHES.items():
    if filename in {"model.safetensors", "config.json"}:
        file_path = os.path.join(
            MODEL_PATH,
            filename,
        )
    else:
        file_path = os.path.join(
            TOKENIZER_PATH,
            filename,
        )

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Required file was not found: {file_path}"
        )

    actual_hash = sha256_file(file_path)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Hash mismatch for {filename}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

    print(f"Hash verified: {filename}")


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# AUDIT DATA
# ============================================================

gate = FilterGate(GROUND_TRUTH)

audited = [
    {
        **record,
        "verdict": gate.audit(record).value,
        "text": fact_text(record),
    }
    for record in SYNTHETIC_DATA
]

all_records = audited

verified_records = [
    record
    for record in audited
    if record["verdict"] == Verdict.VERIFIED.value
]

if not verified_records:
    raise RuntimeError(
        "No verified records were found for HARD_FILTER."
    )

verdict_counts = {
    verdict.value: sum(
        record["verdict"] == verdict.value
        for record in audited
    )
    for verdict in Verdict
}

print("Audited records:", len(audited))
print("Verdict counts:", verdict_counts)


# ============================================================
# RUN EXPERIMENT
# ============================================================

all_runs = []

for seed in SEEDS:
    print("\n" + "=" * 60)
    print(f"STARTING SEED: {seed}")
    print("=" * 60)

    contaminated_stream = cycle_to_length(
        all_records,
        UPDATES_PER_EPOCH,
        seed,
    )

    hard_filter_stream = cycle_to_length(
        verified_records,
        UPDATES_PER_EPOCH,
        seed,
    )

    epistemic_stream = cycle_to_length(
        all_records,
        UPDATES_PER_EPOCH,
        seed,
    )

    branch_results = {
        "CONTAMINATED": train_branch(
            "CONTAMINATED",
            contaminated_stream,
            seed,
        ),
        "HARD_FILTER": train_branch(
            "HARD_FILTER",
            hard_filter_stream,
            seed,
        ),
        "EPISTEMIC": train_branch(
            "EPISTEMIC",
            epistemic_stream,
            seed,
        ),
    }

    margins = {
        branch: float(
            np.mean([
                row["truth_margin"]
                for row in branch_results[branch]
            ])
        )
        for branch in branch_results
    }

    print(f"Seed {seed} margins:", margins)

    all_runs.append({
        "seed": int(seed),
        "margins": margins,
        "results": branch_results,
    })


# ============================================================
# AGGREGATE RESULTS
# ============================================================

branches = [
    "CONTAMINATED",
    "HARD_FILTER",
    "EPISTEMIC",
]

aggregate = {
    branch: {
        "mean_margin": float(
            np.mean([
                run["margins"][branch]
                for run in all_runs
            ])
        ),
        "std_margin": float(
            np.std(
                [
                    run["margins"][branch]
                    for run in all_runs
                ],
                ddof=1,
            )
        ),
    }
    for branch in branches
}


# ============================================================
# SAVE RESULTS
# ============================================================

result = {
    "experiment": EXPERIMENT,
    "status": "completed",
    "seeds": [int(seed) for seed in SEEDS],
    "aggregate": aggregate,
    "runs": all_runs,
    "independent": True,
    "license_suggested": "MIT",
}

output_file = os.path.join(
    OUTPUT_DIR,
    "filter_exp03_results.json",
)

with open(
    output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("\n" + "=" * 60)
print("EXP03 COMPLETE")
print("Output file:", output_file)
print("Aggregate results:")
print(json.dumps(aggregate, indent=2))
print("SHA256:", sha256_file(output_file))
print("=" * 60)


Using device: cpu
Model path: /kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_model/gpt2_model
Tokenizer path: /kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_tokenizer/gpt2_tokenizer
Hash verified: model.safetensors
Hash verified: config.json
Hash verified: vocab.json
Hash verified: merges.txt
Audited records: 14
Verdict counts: {'VERIFIED': 6, 'CONTRADICTED': 6, 'UNKNOWN': 2, 'INVALID': 0}

STARTING SEED: 11


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 11 | CONTAMINATED | Epoch 1/4 | Loss 1.529519
Seed 11 | CONTAMINATED | Epoch 2/4 | Loss 0.791619
Seed 11 | CONTAMINATED | Epoch 3/4 | Loss 0.567820
Seed 11 | CONTAMINATED | Epoch 4/4 | Loss 0.410650


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 11 | HARD_FILTER | Epoch 1/4 | Loss 1.276996
Seed 11 | HARD_FILTER | Epoch 2/4 | Loss 0.517842
Seed 11 | HARD_FILTER | Epoch 3/4 | Loss 0.334885
Seed 11 | HARD_FILTER | Epoch 4/4 | Loss 0.263118


...

Seed 33 | CONTAMINATED | Epoch 1/4 | Loss 1.581770
Seed 33 | CONTAMINATED | Epoch 2/4 | Loss 0.798514
Seed 33 | CONTAMINATED | Epoch 3/4 | Loss 0.586857
Seed 33 | CONTAMINATED | Epoch 4/4 | Loss 0.457240


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | HARD_FILTER | Epoch 1/4 | Loss 1.159053
Seed 33 | HARD_FILTER | Epoch 2/4 | Loss 0.477878
Seed 33 | HARD_FILTER | Epoch 3/4 | Loss 0.339983
Seed 33 | HARD_FILTER | Epoch 4/4 | Loss 0.263500


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | EPISTEMIC | Epoch 1/4 | Loss -4.943276
Seed 33 | EPISTEMIC | Epoch 2/4 | Loss -13.309993
Seed 33 | EPISTEMIC | Epoch 3/4 | Loss -23.711612
Seed 33 | EPISTEMIC | Epoch 4/4 | Loss -28.961325
Seed 33 margins: {'CONTAMINATED': -15.21693164096886, 'HARD_FILTER': -0.6628633563717207, 'EPISTEMIC': 13.849648793538412}

EXP03 COMPLETE
Output file: /kaggle/working/filter_exp03/filter_exp03_results.json
Aggregate results:
{
  "CONTAMINATED": {
    "mean_margin": -15.738675863162468,
    "std_margin": 0.9680577047519693
  },
  "HARD_FILTER": {
    "mean_margin": -0.8094969778839086,
    "std_margin": 0.27784426343322843
  },
  "EPISTEMIC": {
    "mean_margin": 13.207186804877388,
    "std_margin": 0.5658329910950723
  }
}
SHA256: 369c5cec029b792744978a9c81434ff46528be5fee49377c621a5f67c9441c85
